# RAG Config Walkthrough

로컬 Jupyter 환경에서 RAG config를 선택해 실행하고, ingest/retrieve/chat/evaluate 결과를 확인하는 기본 노트북입니다.

## 1. 실험 기준

이 노트북은 코드를 직접 고치는 대신 config 파일을 선택해 RAG 실험을 실행하는 흐름을 보여줍니다.
처음에는 기준 config를 그대로 실행하고, 필요할 때만 다른 YAML로 바꿔 비교합니다.

- 기준 config: `configs/experiments/rag/rag_langchain.yaml`
- 비교 config 예시: `configs/experiments/rag/rag_hybrid.yaml`
- 준실제 문서 검증 config: `configs/experiments/rag/rag_realistic_docs.yaml`
- 평가 질문: `data/rag_sample/eval_questions.csv`
- 주로 바꿔볼 옵션: `rag.splitter.chunk_size`, `rag.splitter.chunk_overlap`, `rag.retriever.method`, `rag.retriever.top_k`, `rag.answerer.provider`

비교 config는 처음부터 꼭 필요하지 않습니다. config 루트는 `CONFIG_ROOT = Path("configs/experiments/rag")`로 고정하고, `EXP_NAME`만 YAML 파일 이름에 맞춰 바꿉니다. 기준안이 없을 때는 `EXP_NAME = "rag_langchain"`으로 산출물 계약이 맞는지 먼저 확인합니다.

| `EXP_NAME` | config | 목적 |
| --- | --- | --- |
| `rag_langchain` | `rag_langchain.yaml` | 가장 먼저 돌리는 기준 실험입니다. TXT 샘플로 RAG 산출물 계약을 확인합니다. |
| `rag_hybrid` | `rag_hybrid.yaml` | retriever 방식 비교가 필요할 때 쓰는 예시 config입니다. 처음부터 필수는 아닙니다. |
| `rag_realistic_docs` | `rag_realistic_docs.yaml` | DOCX/HWPX 준실제 문서에서 loader와 citation metadata가 깨지지 않는지 확인합니다. |

In [ ]:
from pathlib import Path
import json
import os
import shlex
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    """현재 노트북 위치에서 프로젝트 루트 디렉터리를 찾습니다."""
    markers = ("AGENTS.md", "configs", "scripts", "src")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(
        "프로젝트 루트를 찾지 못했습니다. "
        "AGENTS.md, configs/, scripts/, src/가 있는 위치에서 실행해 주세요."
    )


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
os.chdir(PROJECT_ROOT)
PYTHON = shlex.quote(sys.executable)

from src.config import load_config

CONFIG_ROOT = Path("configs/experiments/rag")

# 기준안이 없을 때는 rag_langchain부터 실행합니다.
# 비교가 필요하면 EXP_NAME을 "rag_hybrid" 같은 config 파일 이름으로 바꿉니다.
EXP_NAME = "rag_langchain"
RAG_CONFIG = CONFIG_ROOT / f"{EXP_NAME}.yaml"
COMPARE_CONFIG = CONFIG_ROOT / "rag_hybrid.yaml"
REALISTIC_EXP_NAME = "rag_realistic_docs"
REALISTIC_CONFIG = CONFIG_ROOT / f"{REALISTIC_EXP_NAME}.yaml"

# QUESTION은 retrieve/chat 동작을 눈으로 확인하는 디버깅용 단일 질문입니다.
# 전체 평가는 config의 evaluation.questions_path CSV를 --evaluate로 실행합니다.
QUESTION = "예산은 얼마야?"


def config_output_dir(config_path: str | Path) -> Path:
    """config의 paths.output_dir을 읽어 실험 산출물 위치를 반환합니다."""
    return Path(load_config(config_path)["paths"]["output_dir"])


def config_questions_path(config_path: str | Path) -> Path:
    """config의 evaluation.questions_path를 읽어 평가 질문 위치를 반환합니다."""
    return Path(load_config(config_path)["evaluation"]["questions_path"])


OUTPUT_DIR = config_output_dir(RAG_CONFIG)
REALISTIC_OUTPUT_DIR = config_output_dir(REALISTIC_CONFIG)
EVAL_QUESTIONS = config_questions_path(RAG_CONFIG)

print(f"project root: {PROJECT_ROOT}")
print(f"python: {sys.executable}")
print(f"experiment: {EXP_NAME}")
print(f"config: {RAG_CONFIG}")
print(f"output_dir: {OUTPUT_DIR}")


## OpenAI API key 선택 설정

OpenAI 사용 여부는 노트북 변수가 아니라 config의 `rag.embedding.provider` 또는 `rag.answerer.provider`로 결정합니다.
기본 config는 local provider라 비용이 발생하지 않습니다. OpenAI provider config를 선택했을 때만 아래 환경변수가 필요합니다.

In [ ]:
# OpenAI provider config를 사용할 때만 아래 두 줄의 주석을 해제합니다.
# import getpass
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

print("OPENAI_API_KEY 설정 여부:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
def run(command: str) -> None:
    print(f"$ {command}")
    result = __import__("subprocess").run(command, shell=True, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"command failed: {command}")

def show_json(path: str | Path):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path}")
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def show_csv(path: str | Path, n: int = 5):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path).head(n)

def show_jsonl(path: str | Path, n: int = 3):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path}")
        return []
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return rows[:n]

def display_metrics(path: str | Path):
    metrics = show_json(path)
    if not metrics:
        return pd.DataFrame()
    rows = [{"metric": key, "value": value} for key, value in metrics.items()]
    return pd.DataFrame(rows)

def display_answers(path: str | Path, n: int = 5):
    rows = show_jsonl(path, n=n)
    summary = []
    for row in rows:
        citations = row.get("citations", [])
        citation_text = "; ".join(
            f"{item.get('source_path', '')}:{item.get('chunk_id', '')}"
            for item in citations
        )
        summary.append({
            "question": row.get("question"),
            "status": row.get("status"),
            "answer": row.get("answer"),
            "citations": citation_text,
        })
    return pd.DataFrame(summary)

def display_failure_tables(output_dir: str | Path, n: int = 5):
    output_dir = Path(output_dir)
    tables = {
        "bad_retrievals": output_dir / "bad_retrievals.csv",
        "unsupported_answers": output_dir / "unsupported_answers.csv",
        "failed_questions": output_dir / "failed_questions.csv",
    }
    for title, path in tables.items():
        print(f"## {title}")
        display(show_csv(path, n=n))

def display_artifact_check(output_dir: str | Path):
    output_dir = Path(output_dir)
    required = [
        "config.yaml", "parsed_documents.csv", "chunks.csv", "embeddings.jsonl",
        "retrieval_results.jsonl", "answers.jsonl", "evaluation_results.csv",
        "bad_retrievals.csv", "unsupported_answers.csv", "failed_questions.csv",
        "metrics.json", "run_status.json", "rag_ingest_checkpoint.json",
    ]
    rows = [{"artifact": name, "exists": (output_dir / name).exists()} for name in required]
    rows.append({"artifact": "failure.log", "exists": (output_dir / "failure.log").exists()})
    return pd.DataFrame(rows)

## 2. 입력 확인

원본 문서와 평가 질문 CSV가 준비되어 있는지 확인합니다.
여기서 파일이 없으면 pipeline 문제가 아니라 경로 또는 데이터 준비 문제일 가능성이 큽니다.

`QUESTION`은 단일 질문으로 retrieval/chat 흐름을 빠르게 보는 용도입니다. 실제 평가와 metric 계산은 `EVAL_QUESTIONS` CSV 전체를 대상으로 실행합니다.

`OUTPUT_DIR`과 `EVAL_QUESTIONS`는 config에서 읽어옵니다. 실험 이름을 바꿀 때는 경로 문자열을 여러 곳에서 직접 고치지 말고 위의 `CONFIG_ROOT`와 `EXP_NAME`을 먼저 확인합니다.

In [ ]:
print(RAG_CONFIG, RAG_CONFIG.exists())
print(EVAL_QUESTIONS, EVAL_QUESTIONS.exists())
show_csv(EVAL_QUESTIONS)

## 3. 실행 전 config 점검

`check_rag_pipeline.py`는 실제 산출물을 만들기 전에 config와 입력 경로를 검증합니다.
실패하면 먼저 YAML의 상대 경로가 프로젝트 루트 기준으로 맞는지 확인합니다.

In [ ]:
run(f"{PYTHON} scripts/check_rag_pipeline.py --config {RAG_CONFIG} --project-root .")

## 4. Ingest 실행

문서를 읽고 chunk와 embedding 산출물을 만듭니다.
이 단계의 핵심 산출물은 `parsed_documents.csv`, `chunks.csv`, `embeddings.jsonl`입니다.
chunk가 너무 짧거나 길면 `rag.splitter.chunk_size`와 `rag.splitter.chunk_overlap`을 조정합니다.

In [ ]:
run(f"{PYTHON} scripts/run_rag_ingest.py --config {RAG_CONFIG} --project-root .")
show_csv(OUTPUT_DIR / "chunks.csv")

## 5. 검색 결과 확인

질문 하나를 던져 어떤 chunk가 검색되는지 확인합니다.
답변 품질이 나쁘면 먼저 LLM보다 retrieval 결과가 맞는지 보는 것이 좋습니다.

In [ ]:
run(f"{PYTHON} scripts/run_rag_retrieve.py --config {RAG_CONFIG} --project-root . --question {shlex.quote(QUESTION)}")

## 6. 답변과 평가 실행

먼저 `QUESTION` 하나로 답변과 citation 모양을 확인합니다.
그 다음 `--evaluate`로 config의 `evaluation.questions_path`에 있는 평가 질문 CSV 전체를 실행합니다.

즉 단일 질문은 디버깅용이고, metric은 평가 CSV 전체 실행 결과를 기준으로 봅니다. 평가는 검색 성공 여부, 답변에 기대 답이 포함되는지, citation이 맞는지를 함께 봅니다.

In [ ]:
run(f"{PYTHON} scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --question {shlex.quote(QUESTION)}")
run(f"{PYTHON} scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --evaluate")

## 7. Metric과 실패 사례 확인

`metrics.json`은 표로 요약해서 봅니다.
점수가 낮을 때는 아래 실패 CSV를 보고 retrieval 문제인지, answerer 문제인지 분리해서 판단합니다.

In [ ]:
display_metrics(OUTPUT_DIR / "metrics.json")

In [ ]:
display_failure_tables(OUTPUT_DIR)

## 8. 답변과 citation 확인

답변만 보면 그럴듯해 보여도 근거가 틀릴 수 있습니다.
여기서는 질문, 답변, 상태, citation을 한 표로 보고 답변이 실제 검색 근거를 따르는지 확인합니다.

In [ ]:
display_answers(OUTPUT_DIR / "answers.jsonl")

## 9. Retriever 비교 리포트

비교 config는 있으면 좋지만, 처음부터 반드시 필요하지는 않습니다.
기준안이 없을 때는 `EXP_NAME = "rag_langchain"`으로 먼저 돌려 산출물 계약을 확인합니다.
그 다음 `rag_keyword.yaml`, `rag_langchain.yaml`, `rag_hybrid.yaml`을 한 번에 돌려 검색 방식을 비교합니다.

실제 프로젝트에서는 이 결과를 보고 기본 retriever와 `top_k` 후보를 정합니다.

In [ ]:
run(f"{PYTHON} scripts/compare_rag_retrievers.py --project-root .")
show_csv("reports/rag_retriever_comparison.csv")

## 10. 팀 공유 전 리허설

`RAG_REALISTIC`이라는 이름으로 부르는 흐름은 품질 비교용 baseline이 아니라, TXT 샘플보다 실제 문서에 가까운 DOCX/HWPX fixture를 검증하는 리허설입니다.
즉 `configs/experiments/rag/rag_realistic_docs.yaml`은 실제 외부 RFP가 들어오기 전에 loader, chunk metadata, retrieval, answer, citation 산출물이 깨지지 않는지 확인하는 용도입니다.

팀원에게 설명하기 전에는 아래 셀로 DOCX/HWPX 준실제 샘플을 `check -> ingest -> evaluate` 순서로 확인합니다.

스크립트 하나로 숨기지 않고 실제 실행 명령을 그대로 보여주면, 팀원이 파이프라인 흐름을 이해하기 쉽습니다.

In [ ]:
run(f"{PYTHON} scripts/check_rag_pipeline.py --config {REALISTIC_CONFIG} --project-root .")
run(f"{PYTHON} scripts/run_rag_ingest.py --config {REALISTIC_CONFIG} --project-root .")
run(f"{PYTHON} scripts/run_rag_chat.py --config {REALISTIC_CONFIG} --project-root . --evaluate")

## 11. DOCX/HWPX 준실제 샘플 산출물 확인

실제 RFP 문서는 TXT보다 DOCX/HWPX/PDF처럼 다양한 포맷으로 들어올 가능성이 큽니다. 아래 셀은 준실제 fixture의 metric과 답변/citation을 확인합니다.

In [ ]:
display_metrics(REALISTIC_OUTPUT_DIR / "metrics.json")

In [ ]:
display_artifact_check(REALISTIC_OUTPUT_DIR)

In [ ]:
display_answers(REALISTIC_OUTPUT_DIR / "answers.jsonl")